# Mangrove Fragmentation Segmentation, 1990–2024

Input: binary mangrove classifications (1 = mangrove, 0 = non-mangrove, ~30 m, EPSG:4326), one GeoTIFF per year.

**1. Fragmentation segmentation** (Landscape Fragmentation Tool, Vogt et al. 2007; Parent & Hurd, LFT v2). Each mangrove pixel is labelled:

| Class | Rule |
|---|---|
| **Patch** | fragment too small or thin to hold any core |
| **Edge** | within `EDGE_WIDTH_M` of the outer non-mangrove boundary |
| **Perforated** | within `EDGE_WIDTH_M` of a small enclosed opening (< `GAP_MAX_HA`) |
| **Small / Medium / Large core** | farther than `EDGE_WIDTH_M` from any non-mangrove; split by contiguous core size (< 100 ha, 100–200 ha, > 200 ha) |

**2. Landscape metrics** computed with `pylandstats` and named like R **`landscapemetrics`** (`lsm_c_ca`, `lsm_c_np`, `lsm_c_ed`, …), with the same conventions: 8-neighbour patches, landscape boundary not counted as edge (`count_boundary = FALSE` / `consider_boundary = FALSE`), area in ha, ED in m/ha, PD per 100 ha.

**How to run:** set the folders and parameters below, then run all cells top to bottom. Each processing step shows a progress bar, and a cell refuses to run until the cells it depends on have run with the current parameters. Figures are saved as PNG files in `OUT_DIR/figures/`, the interactive map (MapLibre GL) and Sankey diagrams as HTML in `OUT_DIR/interactive/`; the notebook shows them from those files (nothing is embedded). The last cell exports GeoTIFFs and tables from the results already in memory. The same pipeline runs without Jupyter: `python src/fragmap.py`.

## Parameters

In [ ]:
from pathlib import Path

# Folders (relative to this notebook, or absolute)
DATA_DIR = Path("data/mangrove_1990-2024")  # input: one classification GeoTIFF per year, year in the file name
OUT_DIR = Path("outputs")               # output: figures/, rasters/, tables/, interactive/

# Segmentation (defaults follow LFT, converted to metric)
EDGE_WIDTH_M = 100      # edge depth (m); also used as core edge_depth for tca / ndca
GAP_MAX_HA = 5          # enclosed openings smaller than this are 'perforations'
CORE_SMALL_HA = 100     # core < this = small core
CORE_LARGE_HA = 200     # core >= this = large core

# Metrics
ENN = True              # mean nearest-neighbour distance (~2 min total, parallel); False to iterate faster

In [ ]:
import os
import sys
import matplotlib
matplotlib.use("Agg")  # figures go to files only, never embedded in the notebook
import pandas as pd
from IPython.display import HTML, IFrame, display
sys.path.insert(0, str(Path("src").resolve()))  # fragmap.py lives in src/
import fragmap as fm

pd.set_option("display.float_format", "{:,.3f}".format)
pd.set_option("display.max_columns", 30)


class StepNotRun(Exception):
    """Raised when a cell needs an earlier cell; shown as one red line instead of a traceback."""
    def _render_traceback_(self):
        return [f"\x1b[1;31m⛔ {self}\x1b[0m"]


def _seg_params():
    return dict(edge_width_m=EDGE_WIDTH_M, gap_max_ha=GAP_MAX_HA, core_small_ha=CORE_SMALL_HA, core_large_ha=CORE_LARGE_HA)


# step: (cell to run, check that returns None if ok, or why it must be (re-)run)
_STEPS = {
    "load": ("1.1 Load data", lambda r: "not run yet" if r is None else
             "DATA_DIR changed" if r.data_dir != Path(DATA_DIR).resolve() else None),
    "segmentation": ("1.2 Fragmentation segmentation", lambda r: "not run yet" if not hasattr(r, "seg_params") else
                     "parameters changed" if r.seg_params != _seg_params() else None),
    "metrics": ("1.3 Landscape metrics", lambda r: "not run yet" if not hasattr(r, "metric_params") else
                "parameters changed" if r.metric_params != dict(**_seg_params(), enn=ENN) else None),
}


def require(*steps):
    """Stop this cell unless the given steps have run with the current parameters."""
    r = globals().get("res")
    for step in steps:
        cell, check = _STEPS[step]
        why = check(r)
        if why:
            raise StepNotRun(f"Run the '{cell}' cell first ({why}).")


if "DATA_DIR" not in globals():
    raise StepNotRun("Run the 'Parameters' cell first.")


def _rel(path):
    return Path(os.path.relpath(path)).as_posix()


def show(figs):
    """{name: lambda returning a figure}: render + save each to OUT_DIR/figures (with a progress bar),
    then show them side by side in one row, by file reference (no embedded images)."""
    imgs = "".join(f'<img src="{_rel(p)}" alt="{p.stem}" style="flex:1 1 380px;min-width:0;max-width:100%">'
                   for p in fm.save_figures(figs, OUT_DIR))
    display(HTML(f'<div style="display:flex;flex-wrap:wrap;gap:12px;align-items:flex-start">{imgs}</div>'))


def embed(path, height):
    """Show a saved interactive HTML file (map, Sankey) inside the cell output."""
    display(IFrame(_rel(path), width="100%", height=height))

# 1 · Processing
Run these three cells in order. Everything below reads their results.

### 1.1 Load data

In [ ]:
res = fm.load(DATA_DIR)  # starts fresh: later steps must run again
print(f"Years: {res.years}")
print(f"Grid: {res.profile['width']} x {res.profile['height']} px · pixel {res.px:.1f} m x {res.py:.1f} m ({res.pix_ha:.4f} ha)")
print(f"Landscape area: {res.masks[res.years[0]].size * res.pix_ha:,.0f} ha")

### 1.2 Fragmentation segmentation

In [ ]:
require("load")
res = fm.run_segmentation(res, **_seg_params())

### 1.3 Landscape metrics (mangrove class)
Runs one process per year. Takes about 2 minutes with `ENN = True`.

In [ ]:
require("load", "segmentation")
res = fm.run_metrics(res, enn=ENN)

# 2 · Statistics & visuals
Figures are saved to `OUT_DIR/figures/`, interactive HTML to `OUT_DIR/interactive/`, and shown from there.

### 2.1 Fragmentation maps

In [ ]:
require("load", "segmentation")
show({"maps_all_years": lambda: fm.plot_map_grid(res.segs, res.extent)})

In [ ]:
require("load", "segmentation")
y0, y1 = res.years[0], res.years[-1]
show({f"map_{y0}": lambda: fm.plot_map(res.segs[y0], y0, res.extent),
      f"map_{y1}": lambda: fm.plot_map(res.segs[y1], y1, res.extent)})

### 2.2 Interactive map
MapLibre GL: switch years, overlay the change layer, adjust opacity, change basemap (satellite / OSM). Needs internet for the basemap and the MapLibre library.

In [ ]:
require("load", "segmentation")
html = fm.export_interactive(res, OUT_DIR, only=["fragmentation_map"])
embed(html[0], 640)

### 2.3 Class composition

In [ ]:
require("load", "segmentation")
area = res.composition
share = area.drop(columns="Non-mangrove").pipe(lambda d: d.div(d.sum(axis=1), axis=0) * 100)
display(area.style.format("{:,.0f}").set_caption("Area (ha)"))
display(share.style.format("{:.1f}").background_gradient(cmap="Greens", axis=None).set_caption("Share of mangrove area (%)"))
show({"composition_ha": lambda: fm.plot_composition(res.composition),
      "composition_pct": lambda: fm.plot_composition(res.composition, percent=True)})

### 2.4 Landscape metrics
Column names match `landscapemetrics::lsm_c_<name>`. Definitions and units:

In [ ]:
info = pd.DataFrame(fm.METRIC_INFO, index=["description", "unit"]).T
info.index = "lsm_c_" + info.index
info

In [ ]:
require("load", "segmentation", "metrics")
display(res.metrics.T.style.format("{:,.3f}").set_caption("Class-level metrics per year"))
change = (res.metrics.iloc[-1] / res.metrics.iloc[0] - 1) * 100
display(change.rename(f"% change {res.years[0]}→{res.years[-1]}").to_frame().style.format("{:+.1f}"))
show({"landscape_metrics": lambda: fm.plot_metrics(res.metrics)})

### 2.5 Patch size distribution

In [ ]:
require("load", "segmentation", "metrics")
display(res.patches.groupby("year")["area"].describe().style.format("{:,.2f}").set_caption("Patch area (ha)"))
show({"patch_sizes": lambda: fm.plot_patch_sizes(res.patches)})

### 2.6 Change and class transitions

In [ ]:
require("load", "segmentation")
y0, y1 = res.years[0], res.years[-1]
show({f"change_{y0}_{y1}": lambda: fm.plot_change(res.masks[y0], res.masks[y1], y0, y1, res.extent, res.pix_ha)})
t = fm.transitions(res.segs[y0], res.segs[y1], res.pix_ha)
t.style.format("{:,.0f}").background_gradient(cmap="Blues", axis=None).set_caption(f"Transition matrix {y0} → {y1} (ha)")

### 2.7 Sankey diagrams
Hover a flow or node for hectares. Every column sums to the same area (pixels that were mangrove in at least one year).

In [ ]:
require("load", "segmentation")
y0, y1 = res.years[0], res.years[-1]
html = fm.export_interactive(res, OUT_DIR, only=[f"sankey_{y0}_{y1}"])
embed(html[0], 760)

In [ ]:
require("load", "segmentation")
html = fm.export_interactive(res, OUT_DIR, only=["sankey_all_years"])
embed(html[0], 760)

# 3 · Export
Writes GeoTIFFs (with embedded colour table, open directly in QGIS/ArcGIS) and CSV + multi-sheet Excel (all transition matrices) from the results computed above. Figures and interactive HTML were already saved by section 2.

In [ ]:
require("load", "segmentation", "metrics")
fm.export_rasters(res, OUT_DIR)
fm.export_tables(res, OUT_DIR)
for f in sorted(OUT_DIR.rglob("*.*")):
    print(f)